# GridSearchCV

GridSearchCV falls under **Model Selection & Hyperparameter Tuning**, which is a distinct stage of the ML workflow from Feature Engineering (transformation/selection/extraction/construction) and Missing Value Handling. It's the tool that ties together everything from your **Machine Learning Pipelines** discussion earlier — it's commonly used *on top of* a Pipeline to tune the whole thing (preprocessing + model) at once.

---

## What is GridSearchCV?

`GridSearchCV` (from `sklearn.model_selection`) is a technique for **hyperparameter tuning** — systematically searching through a specified set of hyperparameter combinations to find the one that gives the best model performance, evaluated using **cross-validation**.

The name breaks down as:
- **Grid Search** — exhaustively tries **every possible combination** of hyperparameter values you specify (like a grid of possibilities)
- **CV** — each combination is evaluated using **cross-validation** (not just a single train/test split), for a more reliable performance estimate

---

## Why It's Needed

Every ML model has **hyperparameters** — settings *you* choose before training (not learned from data), like:
- `k` in KNN
- `C` and `kernel` in SVM
- `max_depth`, `n_estimators` in Random Forest
- `alpha` in Ridge/Lasso Regression

Picking these manually by trial and error is slow, unsystematic, and easy to get wrong. GridSearchCV automates this by trying **all combinations** and picking the best one based on a chosen scoring metric.

---

## How It Works

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',      # metric to optimize
    n_jobs=-1                # use all CPU cores
)

grid.fit(X_train, y_train)

print(grid.best_params_)    # best hyperparameter combination
print(grid.best_score_)     # best cross-validated score
print(grid.best_estimator_) # the actual fitted model with best params
```

**Step by step:**
1. You define a `param_grid` — a dictionary of hyperparameters and the values to try for each.
2. GridSearchCV generates **every combination** (Cartesian product) of those values.
   - In the example above: 3 × 3 × 3 = **27 combinations**
3. For **each combination**, it trains and evaluates the model using **k-fold cross-validation** (e.g., 5-fold → data split into 5 parts, trained/tested 5 times, rotating which part is the test fold).
   - Total fits = 27 combinations × 5 folds = **135 model fits**
4. It averages the CV scores for each combination and picks the one with the **best average score**.
5. Finally, it **refits** the best combination on the *entire* training set (`refit=True` by default) to produce the final `best_estimator_`.

---

## Combining GridSearchCV with Pipelines (connects to your earlier pipeline discussion)

This is the powerful part — you can tune **preprocessing steps AND model hyperparameters together**, using the `stepname__parameter` naming convention:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression())
])

param_grid = {
    'power__method': ['yeo-johnson', 'box-cox'],  # tune the transformer choice
    'model__C': [0.01, 0.1, 1, 10],                # tune the model's regularization
    'model__penalty': ['l1', 'l2']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)
```

This is why Pipelines matter so much for GridSearchCV specifically: **each fold's preprocessing is refit only on that fold's training data**, avoiding data leakage — exactly the benefit discussed earlier. Without wrapping preprocessing in a Pipeline, you'd risk leaking test-fold information into your scaler/transformer during tuning.

---

## Key Parameters

| Parameter | Purpose |
|---|---|
| `estimator` | The model (or Pipeline) to tune |
| `param_grid` | Dict (or list of dicts) of hyperparameters and values to try |
| `cv` | Number of cross-validation folds (default 5) |
| `scoring` | Metric to optimize (`'accuracy'`, `'f1'`, `'roc_auc'`, `'neg_mean_squared_error'`, etc.) |
| `n_jobs` | Parallelization — `-1` uses all available CPU cores |
| `refit` | Whether to refit the best model on the full training data (default `True`) |
| `verbose` | Controls how much progress output is printed during the search |

---

## Downsides / Considerations

1. **Computationally expensive** — grows combinatorially. With `p` hyperparameters, each with `n` values, and `k`-fold CV, total fits = n^p × k. This explodes quickly with more parameters/values.
2. **Alternative: `RandomizedSearchCV`** — instead of trying every combination, randomly samples a fixed number of combinations from the grid (or distributions). Much faster for large search spaces, often nearly as good.
3. **Alternative: Bayesian optimization** (e.g., `Optuna`, `scikit-optimize`) — smarter search that uses results of previous trials to guide which combinations to try next, rather than exhaustive/random search.

---